<a href="https://www.kaggle.com/code/insanejsk/crossencoder-reranker-train?scriptVersionId=327439791" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

Cross-Encoder Reranker
---
- Input : reranker_train.jsonl  (query, positive, negative)
- Model : bert-base-uncased → [CLS] query [SEP] passage [SEP] → Linear(768,1) → scalar score
- Loss  : Margin ranking loss: max(0, margin - (score_pos - score_neg))
- Eval  : nDCG@10, MRR  (using bi-encoder v1 -> reranker reorders -> top-10)
- Output: best_reranker/  (saved by best nDCG@10)

In [1]:
import json
import math
import os
import time
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from safetensors.torch import load_file, save_file

In [2]:
CONFIG = {
    "model_name": "bert-base-uncased",
    "max_length": 192,      # query + passage concatenated
    "batch_size": 48,
    "epochs": 1,
    "lr": 2e-5,
    "margin": 1.0,
    "warmup_ratio": 0.1,
    "train_path": "/kaggle/input/datasets/insanejsk/ms-marco-rerankersample/reranker_train.jsonl",
    "val_path": "/kaggle/input/datasets/insanejsk/ms-marco-rerankersample/reranker_val.jsonl",
    "biencoder_path": "/kaggle/input/models/insanejsk/denseretriever/pytorch/default/1/best_biencoder",
    "output_dir": "/kaggle/working/best_reranker/",
    "eval_queries": 500,      # number of val queries for nDCG eval
    "rerank_top_k": 50,       # bi-encoder retrieves, reranker reorders
    "final_top_n": 10,       # returns after reranking
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
N_GPUS = torch.cuda.device_count()
print(f"Using {N_GPUS} GPU(s)" if N_GPUS > 0 else "Using CPU")

Device: cuda
Using 2 GPU(s)


In [3]:
class PairDataset(Dataset):
    """
    Each sample: (query, positive_passage, negative_passage)
    Cross-encoder scores each (query, passage) pair independently.
    """
    def __init__(self, path, max_samples=None):
        self.samples = []
        with open(path) as f:
            for i, line in enumerate(f):
                if max_samples and i >= max_samples:
                    break
                obj = json.loads(line)
                self.samples.append((
                    obj["query"],
                    obj["positive"],
                    obj["negative"],
                ))
        print(f"Loaded {len(self.samples)} triples from {path}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [4]:
def collate_fn(batch, tokenizer, max_length):
    """
    For each triple:
      - encode(query, pos) -> input_ids_pos, attn_pos
      - encode(query, neg) -> input_ids_neg, attn_neg
    Returns tokenized batches: enc_pos, enc_neg
    """
    queries   = [b[0] for b in batch]
    positives = [b[1] for b in batch]
    negatives = [b[2] for b in batch]

    enc_pos = tokenizer(
        queries, positives,
        max_length=max_length,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )
    enc_neg = tokenizer(
        queries, negatives,
        max_length=max_length,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )
    return enc_pos, enc_neg

In [5]:
# Model
class CrossEncoder(nn.Module):
    """
    Takes a single (query, passage) concatenated input and gives a scalar relevance score.
    BERT-base -> pool CLS token -> Linear(768, 1) -> Scalar score
    """
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size  = self.encoder.config.hidden_size   # 768 for bert-base
        self.score_head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        cls_emb = outputs.last_hidden_state[:, 0, :]    # [B, 768]  CLS token
        score   = self.score_head(cls_emb).squeeze(-1)  # [B]
        return score

In [6]:
# Loss
def margin_ranking_loss(score_pos, score_neg, margin=1.0):
    """
    L = mean(max(0, margin - (score_pos - score_neg)))
    """
    return torch.clamp(margin - (score_pos - score_neg), min=0.0).mean()

### nDCG@k (Normalized Discounted Cumulative Gain at k)

nDCG@k is a ranking metric that measures how well the top k predicted items are ordered compared to the ideal ranking. It rewards placing highly relevant items near the top of the list and normalizes the score so that the best possible ranking has a value of 1.

The **Discounted Cumulative Gain (DCG@10)** is:

$$
\mathrm{DCG@10} = \sum_{i=1}^{10} \frac{2^{\mathrm{rel}_i}-1}{\log_2(i+1)}
$$

where $\mathrm{rel}_i$ is the relevance of the item at rank $i$.

The **Ideal DCG (IDCG@10)** is the DCG obtained from the perfectly sorted ranking:

$$
\mathrm{IDCG@10} = \sum_{i=1}^{10} \frac{2^{\mathrm{rel}_i^{*}}-1}{\log_2(i+1)}
$$

Finally,

$$
\mathrm{nDCG@10} = \frac{\mathrm{DCG@10}}{\mathrm{IDCG@10}}
$$

with values in $[0,1]$, where **1** indicates a perfect ranking.

In [7]:
# Metrics
def dcg_at_k(relevances, k):
    """Discounted Cumulative Gain at k."""
    relevances = np.array(relevances[:k])
    if relevances.size == 0:
        return 0.0
    positions = np.arange(1, relevances.size + 1)
    return float(np.sum(relevances / np.log2(positions + 1)))


def ndcg_at_k(relevances, k):
    """Normalized DCG at k. Ideal = single relevant doc at position 1."""
    actual  = dcg_at_k(relevances, k)
    ideal   = dcg_at_k(sorted(relevances, reverse=True), k)
    return actual / ideal if ideal > 0 else 0.0


def mrr(relevances):
    """Mean Reciprocal Rank for single query."""
    for i, r in enumerate(relevances):
        if r > 0:
            return 1.0 / (i + 1)
    return 0.0

In [8]:
# Bi-encoder
class BiEncoder(nn.Module):
    """Follows the architecture of DenseRetriever-v1"""
    def __init__(self, model_name, proj_dim=256):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.proj = nn.Sequential(
            nn.Linear(hidden, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim),
        )
    def encode(self, texts, tokenizer, max_length, device, batch_size=64):
        all_embs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc   = tokenizer(
                batch,
                max_length=max_length,
                padding=True,
                truncation=True,
                return_tensors="pt",
            ).to(device)
            with torch.no_grad():
                out = self.encoder(**enc)
                mask = enc["attention_mask"].unsqueeze(-1).float()
                emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
                emb = self.proj(emb)
                emb = nn.functional.normalize(emb, dim=-1)
            all_embs.append(emb.cpu())
        return torch.cat(all_embs, dim=0)

def load_biencoder(path, device):
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = BiEncoder(path, proj_dim=256)
    proj_state = load_file(os.path.join(path, "proj.safetensors"), device=str(device))
    model.proj.load_state_dict(proj_state)
    model.to(device).eval()
    return model, tokenizer

In [9]:
# Evals
def evaluate(cross_encoder, bi_model, bi_tokenizer, val_samples, config, device):
    """
    For each val query:
      1. bi-encoder retrieves top-K passages from full val corpus
      2. cross-encoder reranks those K passages
      3. compute nDCG@10, MRR on reranked list
    """
    cross_encoder.eval()
    ce_tokenizer = AutoTokenizer.from_pretrained(config["model_name"])

    # all unique passages across val set
    corpus = list({s["positive"] for s in val_samples} | {s["negative"] for s in val_samples})
    corpus_idx = {p: i for i, p in enumerate(corpus)}

    # encode full corpus once with bi-encoder
    print(f"Encoding corpus ({len(corpus)} passages)...")
    corpus_embs = bi_model.encode(
        corpus, bi_tokenizer,
        max_length=180, device=device,
    )  # (C, 256)

    queries = [s["query"] for s in val_samples[:config["eval_queries"]]]
    pos_set = [s["positive"] for s in val_samples[:config["eval_queries"]]]

    ndcg_scores, mrr_scores = [], []

    print(f"Evaluating {len(queries)} queries...")
    for q, pos in zip(queries, pos_set):
        
        # bi-encoder retrieval
        q_emb = bi_model.encode(
            [q], bi_tokenizer, max_length=32, device=device,
            )  # (1, 256)
        sims = (corpus_embs @ q_emb.T).squeeze(-1)  # (C,)
        top_k = sims.topk(config["rerank_top_k"]).indices.tolist()
        cands = [corpus[i] for i in top_k]

        # cross-encoder reranking
        enc = ce_tokenizer(
            [q] * len(cands),
            cands,
            max_length=config["max_length"],
            padding=True,
            truncation=True,
            return_tensors="pt",
            ).to(device)

        with torch.no_grad():
            scores = cross_encoder(
                enc["input_ids"],
                enc["attention_mask"],
                enc.get("token_type_ids"),
            )  # (K,)

        order = scores.argsort(descending=True).tolist()
        reranked = [cands[i] for i in order]

        # Relevance labels (1 if positive, 0 otherwise)
        rels = [1 if p == pos else 0 for p in reranked]

        ndcg_scores.append(ndcg_at_k(rels, config["final_top_n"]))
        mrr_scores.append(mrr(rels))

    return float(np.mean(ndcg_scores)), float(np.mean(mrr_scores))

In [10]:
# Baseline
def compute_baseline(bi_model, bi_tokenizer, val_samples, config, device):
    """
    Computes nDCG@10 and MRR for bi-encoder v1 without reranking
    Uses bi-encoder's cosine similarity ordering of top-K as the ranked list
    """
    # build corpus from val set
    corpus = list({s["positive"] for s in val_samples} | {s["negative"] for s in val_samples})
    print(f"Encoding corpus ({len(corpus)} passages)")
    corpus_embs = bi_model.encode(
        corpus, bi_tokenizer,
        max_length=180, device=device,
    )  # (C, 256)

    queries = [s["query"] for s in val_samples[:config["eval_queries"]]]
    pos_set = [s["positive"] for s in val_samples[:config["eval_queries"]]]

    ndcg_scores, mrr_scores, recall_scores = [], [], []

    for q, pos in zip(queries, pos_set):
        q_emb = bi_model.encode(
            [q], bi_tokenizer, max_length=32, device=device,
        )
        sims = (corpus_embs @ q_emb.T).squeeze(-1)  # (C,)

        top_k  = sims.topk(config["rerank_top_k"]).indices.tolist()
        ranked = [corpus[i] for i in top_k]

        rels   = [1 if p == pos else 0 for p in ranked]

        ndcg_scores.append(ndcg_at_k(rels, config["final_top_n"]))
        mrr_scores.append(mrr(rels))
        # recall@10: did positive appear in top-10?
        recall_scores.append(1.0 if any(rels[:10]) else 0.0)

    bi_ndcg = float(np.mean(ndcg_scores))
    bi_mrr = float(np.mean(mrr_scores))
    bi_recall = float(np.mean(recall_scores))

    print(f"\n{'='*50}")
    print(f"Baseline — Bi-Encoder v1 (no reranking)")
    print(f"Recall@10 : {bi_recall:.4f}")
    print(f"MRR       : {bi_mrr:.4f}")
    print(f"nDCG@10   : {bi_ndcg:.4f}")
    print(f"{'='*50}\n")

    return bi_ndcg, bi_mrr, bi_recall

In [10]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

# datasets
train_dataset = PairDataset(CONFIG["train_path"])
val_samples   = []
with open(CONFIG["val_path"]) as f:
    for line in f:
        val_samples.append(json.loads(line))

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=lambda b: collate_fn(b, tokenizer, CONFIG["max_length"]),
)

# model
model = CrossEncoder(CONFIG["model_name"])
if N_GPUS > 1:
    model = nn.DataParallel(model)
model.to(DEVICE)

bi_model, bi_tokenizer = load_biencoder(CONFIG["biencoder_path"], DEVICE)

# print("\nComputing bi-encoder baseline (before reranker training)...")
# baseline_ndcg, baseline_mrr, baseline_recall = compute_baseline(
#     bi_model, bi_tokenizer, val_samples, CONFIG, DEVICE,
# )

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded 1783707 triples from /kaggle/input/datasets/insanejsk/ms-marco-rerankersample/reranker_train.jsonl


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [11]:
# Helper function
def rerank(query, passages, model_path, device=None):
    """
    Standalone reranking function for pipeline integration.
    Usage:
        ranked = rerank("my query", ["passage1", "passage2", ...], "best_reranker/")
        # returns passages sorted by relevance score, highest first
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = CrossEncoder("bert-base-uncased")
    state = load_file(os.path.join(model_path, "model.safetensors"), device=str(device))
    model.load_state_dict(state)
    model.to(device).eval()

    enc = tokenizer(
        [query] * len(passages),
        passages,
        max_length=256,
        padding=True,
        truncation=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        scores = model(
            enc["input_ids"],
            enc["attention_mask"],
            enc.get("token_type_ids"),
        )

    order = scores.argsort(descending=True).tolist()
    return [(passages[i], scores[i].item()) for i in order]

In [13]:
# Training block

# optimizer + scheduler
total_steps = len(train_loader) * CONFIG["epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps = warmup_steps,
    num_training_steps = total_steps,
)
scaler = torch.amp.GradScaler("cuda")

best_ndcg   = 0.0
history     = []

print(f"\n{'='*50}")
print(f"Cross-Encoder Training")
print(f"Train triples: {len(train_dataset):,}")
print(f"Val queries  : {CONFIG['eval_queries']}")
print(f"Rerank top-K : {CONFIG['rerank_top_k']}")
print(f"Epochs       : {CONFIG['epochs']}")
print(f"Batch size   : {CONFIG['batch_size']}")
print(f"Margin       : {CONFIG['margin']}")
print(f"{'='*50}\n")

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    total_loss  = 0.0
    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{CONFIG['epochs']}")
    for step, (enc_pos, enc_neg) in enumerate(loop):
        # move to device
        enc_pos = {k: v.to(DEVICE) for k, v in enc_pos.items()}
        enc_neg = {k: v.to(DEVICE) for k, v in enc_neg.items()}

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            score_pos = model(**enc_pos)
            score_neg = model(**enc_neg)
            loss = margin_ranking_loss(score_pos, score_neg, CONFIG["margin"])

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)

    # evaluate
    print(f"\nEvaluating epoch {epoch}...")
    core_model = model.module if hasattr(model, "module") else model
    ndcg, mrr_score = evaluate(
        core_model, bi_model, bi_tokenizer,
        val_samples, CONFIG, DEVICE,
    )

    history.append({
        "epoch": epoch,
        "loss": round(avg_loss, 4),
        "ndcg": round(ndcg, 4),
        "mrr": round(mrr_score, 4),
    })

    print(f"\nEpoch {epoch} | Loss: {avg_loss:.4f} | "
          f"nDCG@10: {ndcg:.4f} | MRR: {mrr_score:.4f}")

    if ndcg > best_ndcg:
        best_ndcg = ndcg
        save_model = core_model
        save_model.encoder.save_pretrained(CONFIG["output_dir"])
        tokenizer.save_pretrained(CONFIG["output_dir"])
        save_file(
            save_model.state_dict(),
            os.path.join(CONFIG["output_dir"], "model.safetensors"),
        )
        print(f"New best saved (nDCG@10: {ndcg:.4f})")
    else:
        print(f"No improvement (best: {best_ndcg:.4f})")

print(f"\n{'='*50}")
print(f"Training Complete")
print(f"\nResults Summary")
print(f"{'System':<35} {'Recall@10':>10} {'MRR':>10} {'nDCG@10':>10}")
print(f"{'-'*65}")
print(f"{'BM25':<35} {'0.7740':>10} {'0.5765':>10} {'?':>10}")
print(f"{'Bi-Encoder v1 (no reranking)':<35} "
      f"{baseline_recall:>10.4f} {baseline_mrr:>10.4f} {baseline_ndcg:>10.4f}")
print(f"{'Bi-Encoder v1 + Reranker':<35} "
      f"{'—':>10} {'':>10} {best_ndcg:>10.4f}")
print(f"\nEpoch History:")
for h in history:
    print(f"Epoch {h['epoch']} | Loss: {h['loss']} | "
          f"nDCG@10: {h['ndcg']} | MRR: {h['mrr']}")
print(f"{'='*50}")


Cross-Encoder Training
Train triples: 1,783,707
Val queries  : 500
Rerank top-K : 50
Epochs       : 1
Batch size   : 48
Margin       : 1.0



Epoch 1/1:   0%|          | 0/37161 [00:00<?, ?it/s]/tmp/ipykernel_58/4257251964.py:46: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
Epoch 1/1: 100%|██████████| 37161/37161 [5:34:15<00:00,  1.85it/s, loss=0.0257]  



Evaluating epoch 1...
Encoding corpus (48738 passages)...
Evaluating 500 queries...

Epoch 1 | Loss: 0.2897 | nDCG@10: 0.6906 | MRR: 0.6170


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

New best saved (nDCG@10: 0.6906)

Training Complete

Results Summary
System                               Recall@10        MRR    nDCG@10
-----------------------------------------------------------------
BM25                                    0.7740     0.5765          ?
Bi-Encoder v1 (no reranking)            0.8940     0.3815     0.5025
Bi-Encoder v1 + Reranker                     —                0.6906

Epoch History:
Epoch 1 | Loss: 0.2897 | nDCG@10: 0.6906 | MRR: 0.617


In [15]:
#Proceeding training using mini-epoch evals for early stopping
#This will be proceeding with the model trained for 1 epoch in file version 1

# Baselines computed prior
baseline_ndcg = 0.5025
baseline_mrr = 0.3815
baseline_recall = 0.8940

CONFIG["epoch1_path"] = "/kaggle/input/models/insanejsk/crossencoderreranker/pytorch/default/1/best_reranker"
CONFIG["eval_every"] = 6000
CONFIG["patience"] = 2 # If the model performance degrades this many times in a row, we stop

print("Loading epoch 1 checkpoint...")
model = CrossEncoder(CONFIG["model_name"])
 
state = load_file(
    os.path.join(CONFIG["epoch1_path"], "model.safetensors"),
    device=str(DEVICE),
)
model.load_state_dict(state)
 
if N_GPUS > 1:
    print(f"Using {N_GPUS} GPUs")
    model = nn.DataParallel(model)
model.to(DEVICE)
print("Epoch 1 checkpoint loaded")

Loading epoch 1 checkpoint...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using 2 GPUs
Epoch 1 checkpoint loaded


In [19]:
# Training block

# optimizer + scheduler
total_steps = len(train_loader) * CONFIG["epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps = warmup_steps,
    num_training_steps = total_steps,
)
scaler = torch.amp.GradScaler("cuda")

# training state
prev_ndcg = 0.6906   # epoch 1 result
best_ndcg = 0.6906
patience_count = 0
history        = []
total_loss     = 0.0
eval_history   = []        # (step, ndcg, mrr) for all mid-epoch evals
 
print(f"\n{'='*50}")
print(f"Epoch 2 — Mini-epoch training")
print(f"Total steps : {total_steps:,}")
print(f"Eval every : {CONFIG['eval_every']:,} steps")
print(f"Warmup steps : {warmup_steps:,}")
print(f"Patience : {CONFIG['patience']} degrading evals before stop")
print(f"Must beat : nDCG@10 = {best_ndcg:.4f} (epoch 1)")
print(f"{'='*50}\n")

model.train()
loop = tqdm(train_loader, desc="Epoch 2")
stop_flag = False
 
for step, (enc_pos, enc_neg) in enumerate(loop):
    enc_pos = {k: v.to(DEVICE) for k, v in enc_pos.items()}
    enc_neg = {k: v.to(DEVICE) for k, v in enc_neg.items()}
 
    optimizer.zero_grad()
    with torch.amp.autocast("cuda"):
        score_pos = model(**enc_pos)
        score_neg = model(**enc_neg)
        loss = margin_ranking_loss(score_pos, score_neg, CONFIG["margin"])

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()
 
    total_loss += loss.item()
    loop.set_postfix(loss=f"{loss.item():.4f}")
    
    # mid-epoch eval
    if (step + 1) % CONFIG["eval_every"] == 0:
        core_model = model.module if hasattr(model, "module") else model
        ndcg, mrr_score = evaluate(
            core_model, bi_model, bi_tokenizer,
            val_samples, CONFIG, DEVICE,
        )
        avg_loss = total_loss / (step + 1)
        eval_history.append((step + 1, round(ndcg, 4), round(mrr_score, 4)))
 
        print(f"\nStep {step+1:,}/{total_steps:,} | Loss: {avg_loss:.4f} | nDCG@10: {ndcg:.4f} | MRR: {mrr_score:.4f}")
 
        if ndcg > best_ndcg:
            best_ndcg = ndcg
            patience_count = 0
            save_model = core_model
            save_model.encoder.save_pretrained(CONFIG["output_dir"])
            tokenizer.save_pretrained(CONFIG["output_dir"])
            save_file(
                save_model.state_dict(),
                os.path.join(CONFIG["output_dir"], "model.safetensors"),
            )
            print(f"New best saved (nDCG@10: {ndcg:.4f})")
        else:
            patience_count += 1
            print(f"No improvement (best: {best_ndcg:.4f} | "
                  f"patience: {patience_count}/{CONFIG['patience']})")
            if patience_count >= CONFIG["patience"]:
                print(f"\nEarly stopping — nDCG@10 has not improved "
                      f"for {CONFIG['patience']} consecutive evals.")
                stop_flag = True
                break
 
        prev_ndcg = ndcg
        model.train()   # back to train mode after eval
    if stop_flag:
        break

avg_loss = total_loss / min(step + 1, total_steps)

print(f"\nFull Results Summary:")
print(f"{'System':<35} {'Recall@10':>10} {'MRR':>10} {'nDCG@10':>10}")
print(f"{'-'*65}")
print(f"{'BM25':<35} {'0.7740':>10} {'0.5765':>10} {'?':>10}")
print(f"{'Bi-Encoder v1 (no reranking)':<35} "
      f"{baseline_recall:>10.4f} {baseline_mrr:>10.4f} {baseline_ndcg:>10.4f}")
print(f"{'Bi-Encoder v1 + Reranker (ep1)':<35} "
      f"{'—':>10} {'0.6170':>10} {'0.6906':>10}")
print(f"{'Bi-Encoder v1 + Reranker (ep2)':<35} "
      f"{'—':>10} {'':>10} {best_ndcg:>10.4f}")
print(f"{'='*50}")


Epoch 2 — Mini-epoch training
Total steps : 37,161
Eval every : 6,000 steps
Warmup steps : 3,716
Patience : 2 degrading evals before stop
Must beat : nDCG@10 = 0.6906 (epoch 1)



Epoch 2:   0%|          | 0/37161 [00:00<?, ?it/s]/tmp/ipykernel_58/3585084517.py:50: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
Epoch 2:  16%|█▌        | 5999/37161 [57:42<4:59:17,  1.74it/s, loss=0.1508]

Encoding corpus (48738 passages)...
Evaluating 500 queries...


Epoch 2:  16%|█▌        | 6000/37161 [1:09:40<1871:14:09, 216.18s/it, loss=0.1508]


Step 6,000/37,161 | Loss: 0.1943 | nDCG@10: 0.6591 | MRR: 0.5758
No improvement (best: 0.6906 | patience: 1/2)


Epoch 2:  32%|███▏      | 11999/37161 [2:07:18<4:04:26,  1.72it/s, loss=0.1858]   

Encoding corpus (48738 passages)...
Evaluating 500 queries...

Step 12,000/37,161 | Loss: 0.1950 | nDCG@10: 0.6378 | MRR: 0.5527
No improvement (best: 0.6906 | patience: 2/2)

Early stopping — nDCG@10 has not improved for 2 consecutive evals.


Epoch 2:  32%|███▏      | 11999/37161 [2:19:16<4:52:04,  1.44it/s, loss=0.1858]


Full Results Summary:
System                               Recall@10        MRR    nDCG@10
-----------------------------------------------------------------
BM25                                    0.7740     0.5765          ?
Bi-Encoder v1 (no reranking)            0.8940     0.3815     0.5025
Bi-Encoder v1 + Reranker (ep1)               —     0.6170     0.6906
Bi-Encoder v1 + Reranker (ep2)               —                0.6906
